# LSTNet — Dự báo giá cổ phiếu Apple (AAPL)

Phần **LSTNet** trong đồ án nhóm *Dự báo giá cổ phiếu Apple bằng Machine
Learning & Mạng Nơ-ron*. Notebook này mô tả dữ liệu, quy ước tiền xử lý,
kiến trúc, cách huấn luyện và đánh giá sao cho kết quả **so sánh được** với
các mô hình còn lại (ARIMA, Ridge, LightGBM, Transformer) và dùng được cho
Weighted Average Ensemble.

- Bài báo: Lai et al., *Modeling Long- and Short-Term Temporal Patterns with
  Deep Neural Networks*, SIGIR 2018 —
  <https://arxiv.org/pdf/1703.07015.pdf>
- Quy ước chung của nhóm: `README.md` và `Phan_cong_mo_hinh.md`.


## 1. Quy ước chung (bắt buộc)

| Nội dung | Quy ước |
| :--- | :--- |
| Target chính thức | `nextPrice` — giá đóng cửa ngày t+1 |
| Biến mô hình học | `delta = nextPrice - Price` |
| Khôi phục giá | `nextPrice_dự_đoán = Price + delta_dự_đoán` |
| Chia tập | 80% Train / 20% Test, theo thời gian, **không shuffle** |
| Trong Train | 90% `train_fit` / 10% `train_val` |
| Chuẩn hoá | `StandardScaler`, chỉ `fit` trên `train_fit` |
| Cửa sổ | 30 ngày |
| Seed | 42, 43, 44 — báo cáo trung bình ± độ lệch chuẩn |
| Metric | MAE, RMSE, MAPE, R², Directional Accuracy (tính trên giá USD) |
| Baseline | Naive (`nextPrice = Price`) |


## 2. Cấu trúc dự án

| File | Chức năng |
| :--- | :--- |
| `LSTNet.ipynb` | **Notebook chính**: cấu hình → dữ liệu → model → train 3 seed → đánh giá → vẽ |
| `lstnet_config.py` | Toàn bộ siêu tham số và quy ước dữ liệu (`CONFIG`, `PLOT`) |
| `lstnet_util.py` | `LSTNetInit` — chuyển cấu hình thành object dùng trong pipeline |
| `lstnet_datautil.py` | Đọc CSV, chia tập, chuẩn hoá, tạo cửa sổ `X`, `AR`, `y` |
| `lstnet_model.py` | Kiến trúc LSTNet: CNN1D, GRU, Skip-GRU, nhánh AR |
| `lstnet_metrics.py` | MAE, RMSE, MAPE, R², Directional Accuracy và bảng mean ± std |
| `lstnet_plot.py` | Vẽ lịch sử huấn luyện và `nextPrice` thực tế vs dự đoán |
| `Apple_Stock_ready.csv` | Dữ liệu hiện tại (15 cột, 4.020 dòng) |
| `README.ipynb` | Tài liệu này |

Mặc định notebook **không ghi file nào**: kết quả in trong notebook và biểu đồ hiển thị inline.


## 3. Môi trường

```bash
pip install tensorflow numpy pandas scikit-learn matplotlib jupyter
```

Đã kiểm thử với Python 3.12, TensorFlow 2.21 (Keras 3), numpy/pandas và
scikit-learn mới nhất.

Phần LSTNet **không cần** `statsmodels` (ARIMA) hay `lightgbm` — hai thư viện
đó chỉ dùng cho mô hình của thành viên khác.


## 4. Dữ liệu

File hiện tại: `Apple_Stock_ready.csv` — **4.020 dòng × 15 cột**, từ
04/10/2010 đến 28/09/2026, không có giá trị thiếu.

> Nhóm đặt tên file là `Apple_Stock_ML_ready.csv`. Repo đang có
> `Apple_Stock_ready.csv` với đúng 15 cột tương đương; chỉ cần đổi
> `CONFIG["data"]` nếu dùng file của nhóm.

### Feature sử dụng

Tập feature cơ sở gồm **8 cột**:

```python
["Price", "Open", "High", "Low", "Volume",
 "Return_1", "High_Low_Range", "Open_Close_Change"]
```

- Không dùng `Date` và `NextPrice` làm feature.
- Không dùng `Change_Pct` vì gần như trùng `Return_1`
  (chênh lệch chỉ do làm tròn).
- Không dùng `Year`, `Month`, `Day`, `DayOfWeek` (tập feature mở rộng,
  báo cáo riêng nếu thêm).

### Target

- Cột target trong file: `NextPrice` (bộ đọc tự nhận `nextPrice` /
  `NextPrice`).
- Mô hình **không** học trực tiếp `nextPrice` mà học
  `delta = NextPrice - Price`.


## 5. Chia tập

Chia theo thứ tự thời gian, **không shuffle**:

| Tập | Tỉ lệ | Số dòng | Khoảng thời gian |
| :--- | :---: | ---: | :--- |
| Train | 80% | 3.216 | 04/10/2010 → 14/07/2023 |
| — `train_fit` | 90% Train | 2.894 | 04/10/2010 → 31/03/2022 |
| — `train_val` | 10% Train | 322 | 01/04/2022 → 14/07/2023 |
| Test | 20% | 804 | 17/07/2023 → 28/09/2026 |

- `train_fit`: huấn luyện.
- `train_val`: theo dõi `val_loss`, EarlyStopping, và là nơi tính trọng số
  Ensemble (do thành viên phụ trách Ensemble làm).
- Test: chỉ dùng cho đánh giá cuối cùng.


## 6. Vì sao học `delta` thay vì `nextPrice`?

Giá trong Test vượt xa vùng giá của Train:

| Tập | Khoảng `Price` (USD) |
| :--- | :--- |
| Train | 9,95 → 193,97 |
| Test | 165,00 → 341,07 |

Khoảng **74% số dòng Test** có giá cao hơn mức cao nhất của Train. Với mô hình
cây (LightGBM) điều này gây hỏng cấu trúc; với mạng nơ-ron, scaler fit trên
Train khiến đầu vào Test nằm ngoài vùng đã học.

Vì vậy **cả nhóm** thống nhất cho mô hình học:

```text
delta       = nextPrice - Price          # biến mô hình học
nextPrice^  = Price + delta^             # khôi phục giá trước khi tính metric
```

- `delta` và `nextPrice` **không** được đưa vào ma trận feature.
- Với LSTNet, `delta` được `StandardScaler` fit trên `train_fit`, dự đoán
  xong phải `inverse_transform` rồi cộng với `Price` của ngày cuối cửa sổ.
- Naive baseline tương ứng với `delta = 0`.


## 7. Tiền xử lý và cửa sổ

### Chuẩn hoá

- Feature: `StandardScaler`, **chỉ fit trên `train_fit`** rồi transform cho
  `train_val` và Test.
- `delta`: `StandardScaler` riêng, cũng chỉ fit trên `train_fit`.
- Không fit scaler trên toàn bộ dữ liệu (tránh rò rỉ tương lai).

### Cửa sổ

Mỗi mẫu kết thúc ở ngày `t`:

```text
X   = feature của các ngày [t-29 … t]        # shape (30, 8)
AR  = delta của các ngày [t-7 … t-1]         # shape (7, 1), chỉ dùng quá khứ
y   = delta_t = nextPrice_t - Price_t        # đã scale
Giá dự đoán = Price_t + inverse_transform(y^)
```

- `train_fit`: 2.865 mẫu (mất 29 ngày đầu vì chưa đủ cửa sổ).
- `train_val`: 322 mẫu — dùng 29 ngày cuối `train_fit` làm ngữ cảnh nên
  **không mất mẫu đầu**.
- Test: 804 mẫu — dùng 29 ngày cuối `train_val` làm ngữ cảnh.
- Cửa sổ chỉ chứa dữ liệu quá khứ, không bao giờ chứa ngày tương lai.

Kích thước thực tế:

```text
X  train_fit (2865, 30, 8)   AR (2865, 7, 1)   y (2865,)
X  train_val (322, 30, 8)    AR (322, 7, 1)    y (322,)
X  test      (804, 30, 8)    AR (804, 7, 1)    y (804,)
```


## 8. Kiến trúc LSTNet

```text
Input feature X: (30, 8)
│
├── Conv1D(32, kernel=6, relu) → Dropout(0.2)
│        → (batch, 25, 32)
│
├── GRU(64, relu) → Dropout(0.2) ──────────────────────────┐
│                                                          │
├── Skip-GRU (skip = 5 ngày):                              │
│     PreSkipTrans (25 → 5 bước × 5 nhóm)                   ├── Concat → Dense(1) ─┐
│     GRU(32, relu)                                         │                    │
│     PostSkipTrans                                         │                    ├── Add → delta^
│                                                          │                    │
Input delta quá khứ: (7, 1) → Flatten → Dense(1) ───────────┴────────────────────┘
```

- **CNN1D** bắt mẫu cục bộ ngắn hạn giữa các feature.
- **GRU** học phụ thuộc dài hạn trên toàn cửa sổ.
- **Skip-GRU** nối các điểm cách nhau 5 ngày (một tuần giao dịch) để bắt
  tính chu kỳ; số bước skip được tính từ độ dài sau CNN.
- **AR (highway)** là hồi quy tuyến tính trên 7 giá trị `delta` quá khứ đã
  biết, cộng trực tiếp vào đầu ra phi tuyến.
- Đầu ra là **một giá trị `delta` đã scale** cho ngày cuối cửa sổ.

### Siêu tham số mặc định

| Khoá | Giá trị |
| :--- | ---: |
| `window` | 30 |
| `CNNFilters` / `CNNKernel` | 32 / 6 |
| `GRUUnits` / `SkipGRUUnits` | 64 / 32 |
| `skip` | 5 |
| `dropout` | 0.2 |
| `highway` | 7 |
| `lr` / `loss` | 1e-3 / MSE |
| `batchsize` | 64 |
| `epochs` | 200 (EarlyStopping) |
| `earlystop_patience` | 15 |


## 9. Huấn luyện và đánh giá

- **Loss:** MSE trên `delta` đã scale.
- **Optimizer:** Adam, `lr = 1e-3`.
- **EarlyStopping:** theo `val_loss`, `patience = 15`,
  `restore_best_weights = True`.
- **Không shuffle** khi chia tập; thứ tự thời gian được giữ nguyên.
- **3 seed:** 42, 43, 44. Mỗi seed dựng lại model từ đầu.
- Dự đoán cuối cùng cho Ensemble là **trung bình `nextPrice` của 3 seed**.
- Báo cáo metric của từng seed và **mean ± std** qua các seed.

### Metric (tính trên giá USD)

| Metric | Công thức / ý nghĩa |
| :--- | :--- |
| MAE | `mean(|nextPrice - nextPrice^|)` |
| RMSE | `sqrt(mean((nextPrice - nextPrice^)²))` |
| MAPE | `mean(|nextPrice - nextPrice^| / nextPrice) × 100` |
| R² | Hệ số xác định, chỉ mang tính tham khảo vì giá có xu hướng |
| DirAcc | Tỉ lệ dự đoán đúng hướng tăng/giảm so với `Price` hôm nay |

### Naive baseline

| MAE | RMSE | MAPE (%) |
| ---: | ---: | ---: |
| 2,62 | 3,84 | 1,14 |

Mô hình chỉ có giá trị thực tế nếu **thấp hơn** mốc này (ít nhất là MAE/RMSE)
hoặc có DirAcc cao hơn đáng kể.


## 10. Cách chạy

Mở `LSTNet.ipynb`, chạy lần lượt từ trên xuống:

| Bước | Nội dung |
| :---: | :--- |
| 1 | Import thư viện |
| 2 | Cấu hình (sửa `seeds`, `epochs`, …) |
| 3 | Nạp dữ liệu, in bảng chia tập và shape cửa sổ |
| 4 | Dựng model và xem `summary()` |
| 5 | Train từng seed + EarlyStopping + dự đoán |
| 6 | Bảng metric từng seed và mean ± std |
| 7 | So sánh với Naive, tính dự đoán trung bình 3 seed |
| 8–10 | Vẽ lịch sử train, validation, test |

Một số điều chỉnh nhanh trong cell cấu hình:

| Mục đích | Thiết lập |
| :--- | :--- |
| Train lại từ đầu | Restart Kernel → Run All |
| Chỉ chạy 1 seed | `"seeds": [42]` |
| Lưu ảnh PNG | `PLOT["save_plot"] = "save/lstnet"` |
| Tắt biểu đồ | `PLOT["history"] = False`, `PLOT["prediction"] = False` |
